![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None) 
cc_apps.head()

### 1. Handle missing values

In [ ]:
# Replace '?' with NaN
cc_apps_nans_replaced = cc_apps.replace("?", np.nan)

# Impute: mean for numeric columns, most frequent value for object columns
cc_apps_imputed = cc_apps_nans_replaced.copy()
for col in cc_apps_imputed.columns:
    if cc_apps_imputed[col].dtype == "object":
        cc_apps_imputed[col] = cc_apps_imputed[col].fillna(cc_apps_imputed[col].value_counts().index[0])
    else:
        cc_apps_imputed[col] = cc_apps_imputed[col].fillna(cc_apps_imputed[col].mean())

print(cc_apps_imputed.isna().sum().sum())

### 2. Encode categorical features

In [ ]:
cc_apps_encoded = pd.get_dummies(cc_apps_imputed, drop_first=True)
cc_apps_encoded.head()

### 3. Split features and target, then train/test split

In [ ]:
X = cc_apps_encoded.iloc[:, :-1].values
y = cc_apps_encoded.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### 4. Scale features

In [ ]:
scaler = StandardScaler()
rescaledX_train = scaler.fit_transform(X_train)
rescaledX_test = scaler.transform(X_test)

### 5. Baseline logistic regression

In [ ]:
logreg = LogisticRegression()
logreg.fit(rescaledX_train, y_train)

y_train_pred = logreg.predict(rescaledX_train)
print(confusion_matrix(y_train, y_train_pred))

### 6. Hyperparameter tuning with GridSearchCV

In [ ]:
tol = [0.01, 0.001, 0.0001]
max_iter = [100, 150, 200]
param_grid = dict(tol=tol, max_iter=max_iter)

grid_model = GridSearchCV(estimator=logreg, param_grid=param_grid, cv=5)
grid_model_result = grid_model.fit(rescaledX_train, y_train)

best_train_score, best_train_params = grid_model_result.best_score_, grid_model_result.best_params_
print("Best: %f using %s" % (best_train_score, best_train_params))

### 7. Evaluate best model on the test set

In [ ]:
best_model = grid_model_result.best_estimator_
best_score = best_model.score(rescaledX_test, y_test)
print("Accuracy of logistic regression classifier: ", best_score)